In [1]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
import numpy as np

#### Load the data after processing

In [2]:
# data = np.load("radar_features_filtered_manual.npz")
data = np.load("radar_features_filtered.npz")

X = data["X"]
y = data["y"]
u = data["u"]

scaler = StandardScaler()
X = scaler.fit_transform(X)

le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [3]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y_encoded,
    test_size=0.3,
    random_state=42,
    stratify=y_encoded
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

#### Tuned parameters for XGBoost

In [4]:
xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="multi:softmax",
    num_class=len(np.unique(y)),
    eval_metric="mlogloss",
    tree_method="hist"
)

In [5]:
xgb.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softmax'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fro

#### Validation and Testing

In [6]:
val_pred = xgb.predict(X_val)
print("VALIDATION")
print(classification_report(y_val, val_pred))

VALIDATION
              precision    recall  f1-score   support

           0       0.89      0.89      0.89        19
           1       0.86      0.86      0.86        14
           2       0.95      0.98      0.96        41
           3       0.67      0.80      0.73        10
           4       1.00      0.33      0.50         3
           5       0.70      0.64      0.67        11

    accuracy                           0.87        98
   macro avg       0.85      0.75      0.77        98
weighted avg       0.87      0.87      0.86        98



In [7]:
test_pred = xgb.predict(X_test)
print("TEST")
print(classification_report(y_test, test_pred))

TEST
              precision    recall  f1-score   support

           0       0.90      0.90      0.90        20
           1       0.92      0.86      0.89        14
           2       0.91      0.98      0.94        41
           3       0.88      0.70      0.78        10
           4       0.50      0.50      0.50         2
           5       0.92      0.92      0.92        12

    accuracy                           0.90        99
   macro avg       0.84      0.81      0.82        99
weighted avg       0.90      0.90      0.90        99



In [8]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    xgb,
    X,
    y_encoded,
    cv=cv,
    scoring="f1_macro"
)

print("CV scores:", scores)
print("Mean F1:", scores.mean())

CV scores: [0.95688339 0.81629978 0.90454677 0.8597076  0.83645903]
Mean F1: 0.8747793146248963


#### Testing the model with user-wise split and comparing to random split

In [9]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)

train_idx, test_idx = next(gss.split(X, y_encoded, groups=u))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

In [10]:
xgb.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softmax'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fro

In [11]:
test_pred = xgb.predict(X_test)
print("TEST")
print(classification_report(y_test, test_pred))

TEST
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       0.76      0.88      0.81        25
           2       0.69      0.92      0.79        12
           3       0.50      0.56      0.53         9
           4       0.00      0.00      0.00         8
           5       0.62      0.48      0.54        21

    accuracy                           0.64        75
   macro avg       0.43      0.47      0.44        75
weighted avg       0.60      0.64      0.61        75



/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  

In [12]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

logo = LeaveOneGroupOut()

acc_scores = []
f1_scores = []

for train_idx, test_idx in logo.split(X, y_encoded, groups=u):

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

    xgb.fit(X_train, y_train)

    y_pred = xgb.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')

    acc_scores.append(acc)
    f1_scores.append(f1)

    print(f"Fold Accuracy: {acc:.4f} | Fold F1: {f1:.4f}")

print("\n========== FINAL ==========")
print(f"Mean Accuracy: {np.mean(acc_scores):.4f}")
print(f"Mean Macro F1: {np.mean(f1_scores):.4f}")
print(f"Std Accuracy: {np.std(acc_scores):.4f}")
print(f"Std Macro F1: {np.std(f1_scores):.4f}")

Fold Accuracy: 0.5556 | Fold F1: 0.4030
Fold Accuracy: 0.5882 | Fold F1: 0.2872
Fold Accuracy: 0.4286 | Fold F1: 0.3022
Fold Accuracy: 0.2069 | Fold F1: 0.1383
Fold Accuracy: 1.0000 | Fold F1: 1.0000
Fold Accuracy: 0.5800 | Fold F1: 0.4324
Fold Accuracy: 1.0000 | Fold F1: 1.0000
Fold Accuracy: 0.7273 | Fold F1: 0.4982
Fold Accuracy: 1.0000 | Fold F1: 1.0000
Fold Accuracy: 0.5000 | Fold F1: 0.3333
Fold Accuracy: 1.0000 | Fold F1: 1.0000
Fold Accuracy: 0.4928 | Fold F1: 0.1990
Fold Accuracy: 0.8333 | Fold F1: 0.8319
Fold Accuracy: 0.6667 | Fold F1: 0.4000
Fold Accuracy: 0.6875 | Fold F1: 0.5527
Fold Accuracy: 0.7778 | Fold F1: 0.6190
Fold Accuracy: 0.6512 | Fold F1: 0.5594
Fold Accuracy: 0.7059 | Fold F1: 0.2438
Fold Accuracy: 0.6875 | Fold F1: 0.6205
Fold Accuracy: 0.8333 | Fold F1: 0.4545
Fold Accuracy: 0.4444 | Fold F1: 0.1212
Fold Accuracy: 0.0000 | Fold F1: 0.0000

========== FINAL ==========
Mean Accuracy: 0.6530
Mean Macro F1: 0.4998
Std Accuracy: 0.2487
Std Macro F1: 0.2988
